## Extract Text from xmopipe dataset

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook
%matplotlib inline

import sys
import torch
import matplotlib
import matplotlib.pyplot as plt
from human_body_prior.tools.omni_tools import copy2cpu as c2c
import os
os.environ['PYOPENGL_PLATFORM'] = 'egl'
import json
from pathlib import Path

import re
import shutil
import codecs as cs
import pandas as pd
import numpy as np
from tqdm import tqdm
from os.path import join as pjoin
import random

/home/natsalaz/anaconda3/envs/torch_render/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
json_paths = []
for root, _, files in os.walk("../5-Merge/videosPPmerged"):
    for name in files:
        if name.endswith(".json"):
            json_paths.append(os.path.join(root, name))

output_dir = "./XmoPipe/texts"
os.makedirs(output_dir, exist_ok=True)


def extract_actions(actions_text):
    actions = {}
    pos = 0
    while pos < len(actions_text):
        match = re.search(
            r"<(\d+)>(.*?)((?=<\d+>)|(?=$))", actions_text[pos:], re.DOTALL
        )
        if not match:
            break
        pid = match.group(1)
        action_text = match.group(2).strip().replace("\n", " ").replace("  ", " ")
        action_text = re.sub(r"</\d+>", "", action_text)  # remove closing tags
        if action_text:
            actions[pid] = action_text
        pos += match.end()
    return actions


for json_path in json_paths:
    with open(json_path, "r") as f:
        data = json.load(f)
    for key, text in data.items():
        match = re.search(r"<Action>(.*?)</Action>", text, re.DOTALL)
        if not match:
            continue

        actions = extract_actions(match.group(1))
        if not actions:
            continue

        base_name = key.replace("/", "_")
        if base_name.startswith("video_"):
            base_name = base_name[len("video_") :]
        for pid, desc in actions.items():
            out_name = f"{base_name}_{pid}.txt"
            out_path = os.path.join(output_dir, out_name)
            with open(out_path, "w") as out_f:
                out_f.write(desc)

print(f"Action texts extracted and saved to {output_dir}")
print(actions)

Action texts extracted and saved to ./XmoPipe/texts
{'0': 'A person is stretching their torso sideways, hand resting on the knee.', '1': 'A person is performing a seated side bend, hand on the mat for support.'}


## Extract Metadatas from Xmo Dataset

In [3]:
input_root = "../5-Merged/videosPPmerged"
output_dir = "./XmoPipe/metadatas"
os.makedirs(output_dir, exist_ok=True)
for root, dirs, files in os.walk(input_root):
    for name in files:
        if name == "metadata.txt":
            src_path = os.path.join(root, name)
            video_id = os.path.basename(root) 
            out_name = f"{video_id}.txt"
            out_path = os.path.join(output_dir, out_name)
            shutil.copy(src_path, out_path)

print(f"Metadatas extracted and saved to {output_dir}")

Metadatas extracted and saved to ./XmoPipe/metadatas


## Extract Poses from Xmo Dataset

### Please remember to download the following subdataset from AMASS website: https://amass.is.tue.mpg.de/download.php. Note only download the <u>SMPL+H G</u> data.
* ACCD (ACCD)
* HDM05 (MPI_HDM05)
* TCDHands (TCD_handMocap)
* SFU (SFU)
* BMLmovi (BMLmovi)
* CMU (CMU)
* Mosh (MPI_mosh)
* EKUT (EKUT)
* KIT  (KIT)
* Eyes_Janpan_Dataset (Eyes_Janpan_Dataset)
* BMLhandball (BMLhandball)
* Transitions (Transitions_mocap)
* PosePrior (MPI_Limits)
* HumanEva (HumanEva)
* SSM (SSM_synced)
* DFaust (DFaust_67)
* TotalCapture (TotalCapture)
* BMLrub (BioMotionLab_NTroje)

### Unzip all datasets. In the bracket we give the name of the unzipped file folder. Please correct yours to the given names if they are not the same.

### Place all files under the directory **./amass_data/**. The directory structure shoud look like the following:  
./amass_data/  
./amass_data/ACCAD/  
./amass_data/BioMotionLab_NTroje/  
./amass_data/BMLhandball/  
./amass_data/BMLmovi/   
./amass_data/CMU/  
./amass_data/DFaust_67/  
./amass_data/EKUT/  
./amass_data/Eyes_Japan_Dataset/  
./amass_data/HumanEva/  
./amass_data/KIT/  
./amass_data/MPI_HDM05/  
./amass_data/MPI_Limits/  
./amass_data/MPI_mosh/  
./amass_data/SFU/  
./amass_data/SSM_synced/  
./amass_data/TCD_handMocap/  
./amass_data/TotalCapture/  
./amass_data/Transitions_mocap/  

**Please make sure the file path are correct, otherwise it can not succeed.**

In [4]:
# Choose the device to run the body model on.
comp_device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

In [5]:
from human_body_prior.body_model.body_model import BodyModel

male_bm_path = './body_models/smplh/male/model.npz'
male_dmpl_path = './body_models/dmpls/male/model.npz'

female_bm_path = './body_models/smplh/female/model.npz'
female_dmpl_path = './body_models/dmpls/female/model.npz'

num_betas = 10 # number of body parameters
num_dmpls = None # number of DMPL parameters

male_bm = BodyModel(bm_fname=male_bm_path, num_betas=num_betas, num_dmpls=num_dmpls, dmpl_fname=male_dmpl_path).to(comp_device)
faces = c2c(male_bm.f)

female_bm = BodyModel(bm_fname=female_bm_path, num_betas=num_betas, num_dmpls=num_dmpls, dmpl_fname=female_dmpl_path).to(comp_device)

In [6]:
paths = []
folders = []
dataset_names = []

for root, dirs, files in os.walk("../5-Merge/videosPPmerged"):
    folders.append(root)
    for name in files:
        if name.endswith(".npz"):
            dataset_name = root.split("/")[2]
            if dataset_name not in dataset_names:
                dataset_names.append(dataset_name)
            paths.append(os.path.join(root, name))

print(paths[:10])

['../5-Merge/videosPPmerged/video_001/video_001_merged_scene_2.npz']


In [7]:
save_root = './pose_data'
save_folders = [folder.replace("../5-Merge/videosPPmerged", "./pose_data") for folder in folders]
for folder in save_folders:
    os.makedirs(folder, exist_ok=True)
group_path = [[path for path in paths if name in path] for name in dataset_names]
print(group_path)

[['../5-Merge/videosPPmerged/video_001/video_001_merged_scene_2.npz']]


In [8]:
ex_fps = 30
index_path = "./index_xmo.csv"


def xmo_to_pose(src_path, save_path, index_file, lock=None):
    data = np.load(src_path, allow_pickle=True)
    for bkey in data.keys():

        final_name = save_path + "_" + bkey
        if os.path.exists(final_name + ".npy"):
            # print(final_name + ".npy already exists")
            fps = data[bkey].item().get("fps", ex_fps)
            return fps, 0, 0
        bdata = data[bkey].item()
        fps = 0
        try:
            fps = bdata["fps"]
            frame_number = bdata["trans"].shape[0]
        except:
            print(list(bdata.keys()))
            return fps, 0, 0

        fId = 0
        pose_seq = []

        if bdata["gender"] == "male":
            bm = male_bm
        elif bdata["gender"] == "female":
            bm = female_bm
        else:
            coin = random.random()
            #print(coin)
            if coin <= 0.3:
                bm =female_bm
                #print("female")
            else: 
                bm=male_bm
                #print("male")
        down_sample = int(fps / ex_fps)

        bdata_poses = bdata["poses"][::down_sample, ...]
        bdata_trans = bdata["trans"][::down_sample, ...]

        body_parms = {
            "root_orient": torch.Tensor(bdata_poses[:, :3]).to(comp_device),
            "pose_body": torch.Tensor(bdata_poses[:, 3:66]).to(comp_device),
            "pose_hand": torch.Tensor(bdata_poses[:, 66 : 66 + 90]).to(comp_device),
            "trans": torch.Tensor(bdata_trans).to(comp_device),
            "betas": torch.Tensor(
                np.repeat(
                    bdata["betas"][0, :][:num_betas][np.newaxis],
                    repeats=len(bdata_trans),
                    axis=0,
                )
            ).to(comp_device),
        }

        with torch.no_grad():
            body = bm(**body_parms)
        pose_seq_np = body.Jtr.detach().cpu().numpy()
        match_row = index_file[index_file["source_path"] == final_name + ".npy"]

        if match_row.empty:
            return 0, 1, 0

        new_name = match_row.iloc[0]["new_name"]
        txt_path = os.path.join("./XmoPipe/texts", new_name.replace(".npy", ".txt"))
        if not os.path.exists(txt_path):
            # Remove row from CSV with lock
            if lock:
                with lock:
                    index_file = pd.read_csv(index_path)
                    index_file = index_file[
                        index_file["source_path"] != final_name + ".npy"
                    ]
                    index_file.to_csv(index_path, index=False)
            else:
                index_file = index_file[
                    index_file["source_path"] != final_name + ".npy"
                ]
                index_file.to_csv(index_path, index=False)
            return 0, 0, 1

        np.save((save_path + "_" + bkey), pose_seq_np)
    return fps, 0, 0


def _process_file(args):
    path, lock = args
    save_path = path.replace("../5-Merge/videosPPmerged", "./pose_data")[:-4]
    return xmo_to_pose(path, save_path, index_path, lock)

In [9]:
all_count = sum([len(paths) for paths in group_path])
cur_count = 0

This will take a few hours for all datasets, here we take one dataset as an example

To accelerate the process, you could run multiple scripts like this at one time.

In [10]:
import numpy as np
import pandas as pd
import torch
import os
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import Manager
from tqdm import tqdm
import time

In [11]:
print(group_path[0])

['../5-Merge/videosPPmerged/video_001/video_001_merged_scene_2.npz']


In [12]:
index_file = pd.read_csv(index_path)

for paths in group_path:
    dataset_name = paths[0].split("/")[2]
    pbar = tqdm(paths)
    pbar.set_description("Processing: %s" % dataset_name)
    fps = 0
    for path in pbar:
        save_path = path.replace("../5-Merge/videosPPmerged", "./pose_data")[:-4]
        fps = xmo_to_pose(path, save_path, index_file=index_file)
    cur_count += len(paths)
    print("Processed / All (fps %d): %d/%d" % (fps, cur_count, all_count))

Processing: videosPPmerged: 100%|██████████| 1/1 [00:00<00:00, 785.74it/s]


TypeError: %d format: a number is required, not tuple

The above code will extract poses from **XmoPipe** dataset, and put them under directory **"./pose_data"**

## Segment, Mirror and Relocate Motions

In [13]:
import codecs as cs
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from os.path import join as pjoin

In [14]:
def swap_left_right(data):
    assert len(data.shape) == 3 and data.shape[-1] == 3
    data = data.copy()
    data[..., 0] *= -1
    right_chain = [2, 5, 8, 11, 14, 17, 19, 21]
    left_chain = [1, 4, 7, 10, 13, 16, 18, 20]
    left_hand_chain = [22, 23, 24, 34, 35, 36, 25, 26, 27, 31, 32, 33, 28, 29, 30]
    right_hand_chain = [43, 44, 45, 46, 47, 48, 40, 41, 42, 37, 38, 39, 49, 50, 51]
    tmp = data[:, right_chain]
    data[:, right_chain] = data[:, left_chain]
    data[:, left_chain] = tmp
    if data.shape[1] > 24:
        tmp = data[:, right_hand_chain]
        data[:, right_hand_chain] = data[:, left_hand_chain]
        data[:, left_hand_chain] = tmp
    return data

In [15]:
save_dir = './joints'
os.makedirs(save_dir, exist_ok=True)

index_file = pd.read_csv(index_path)
total_amount = index_file.shape[0]
fps = 30

In [16]:

filtered_index_file = index_file[index_file["source_path"].apply(os.path.exists)]
print("Fichiers gardés :", len(filtered_index_file))
missing = index_file[~index_file["source_path"].apply(os.path.exists)]
print("Fichiers manquants :", len(missing))

Fichiers gardés : 2
Fichiers manquants : 0


In [17]:
for i in tqdm(range(total_amount)):
    if i not in filtered_index_file.index:
        continue 
    source_path = filtered_index_file.loc[i]["source_path"]
    new_name = filtered_index_file.loc[i]["new_name"]
    data = np.load(source_path)
    data_m = swap_left_right(data)
    np.save(pjoin(save_dir, new_name), data)
    np.save(pjoin(save_dir, 'M'+new_name), data_m)

100%|██████████| 2/2 [00:00<00:00, 414.95it/s]


In [18]:
for i in tqdm(range(total_amount)):
    if i not in filtered_index_file.index:
        continue
    new_name = filtered_index_file.loc[i]["new_name"]
    source_path = "./XmoPipe/texts/" + new_name[:-4] + ".txt"
    dest_path = "./XmoPipe/texts/M" + new_name[:-4] + ".txt"

    replacements = {
        "left": "right",
        "right": "left",
        "clockwise": "counter-clockwise",
        "counter-clockwise": "clockwise",
    }
    with open(source_path, "r", encoding="utf-8") as f:
        text = f.read()

    for old, new in replacements.items():
        text = text.replace(old, new)

    with open(dest_path, "w", encoding="utf-8") as f:
        f.write(text)

100%|██████████| 2/2 [00:00<00:00, 1414.37it/s]
